# Barcelona — Listings Analysis
**Input:** `../../Data/interim/barcelona_listings_clean.parquet`  
**Prerequisite:** run `01_cleaning.ipynb` first.

Covers univariate, bivariate, and multivariate analysis of cleaned listings.

In [1]:
import sys
sys.path.insert(0, "../..")

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import f_oneway, ttest_ind
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

from airbnb_iip.data.cleaning import (
    clean_listings, missing_report,
    standardize_property_type, rate_bucket,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 80)
pd.set_option("display.max_colwidth", 200)

## 1 · Load cleaned listings

In [2]:
l_df = pd.read_parquet("../../Data/interim/barcelona_listings_clean.parquet")
print(f"Shape: {l_df.shape}")
l_df.head()

Shape: (18862, 79)


,id,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,host_name,host_since,host_location,host_about,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_verifications,host_has_profile_pic,host_identity_verified,neighbourhood_cleansed,neighbourhood_group_cleansed,latitude,longitude,property_type,room_type,accommodates,bedrooms,beds,amenities,price,minimum_nights,maximum_nights,minimum_minimum_nights,maximum_minimum_nights,minimum_maximum_nights,maximum_maximum_nights,minimum_nights_avg_ntm,maximum_nights_avg_ntm,has_availability,availability_30,availability_60,availability_90,availability_365,calendar_last_scraped,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,availability_eoy,number_of_reviews_ly,estimated_occupancy_l365d,estimated_revenue_l365d,first_review,last_review,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month,bathrooms_number,bathrooms_description,host_tenure_years,days_since_first_review,days_since_last_review,review_span_years,property_type_std,host_response_rate_cat,host_acceptance_rate_cat,price_cat,description_length
0,30320,20250914152907,2025-09-15,city scrape,Apartamentos Dana Sol,None,Unknown,https://a0.muscache.com/pictures/336868/f67409fb_original.jpg,130907,Danuta Weronika,2010-05-24,"Barcelona, Spain","Apartasol offers a network of several spacious, comfortable apartments in the vibrant heart of Barcelona. The apartments are situated in three different locations, all very central and within walkin...",within an hour,1.0,1.00,False,"['email', 'phone']",True,True,Sol,Centro,40.41476,-3.70418,Entire rental unit,Entire home/apt,2,1,2,"[""TV with standard cable"", ""Elevator"", ""Air conditioning"", ""Kitchen"", ""Heating"", ""Wifi""]",157.0,5,50,1.0,7.0,50.0,50.0,5.0,50.0,True,16,46,76,342,2025-09-15,173,1,1,88,0,10,1570.0,2010-07-06,2025-08-27,4.63,4.71,4.88,4.82,4.78,4.90,4.69,Unknown,True,17,17,0,0,0.93,1.0,bath,15.312799,5550.0,19.0,15.143053,Entire place,high,high,high,0
1,40916,20250914152907,2025-09-15,city scrape,Apartasol Apartamentos Dana,None,Unknown,https://a0.muscache.com/pictures/hosting/Hosting-U3RheVN1cHBseUxpc3Rpbmc6NDA5MTY=/original/836b5906-af73-4491-9790-67af0f439d11.jpeg,130907,Danuta Weronika,2010-05-24,"Barcelona, Spain","Apartasol offers a network of several spacious, comfortable apartments in the vibrant heart of Barcelona. The apartments are situated in three different locations, all very central and within walkin...",within an hour,1.0,1.00,False,"['email', 'phone']",True,True,Universidad,Centro,40.42247,-3.70577,Entire rental unit,Entire home/apt,2,1,3,"[""Elevator"", ""Wifi"", ""Air conditioning"", ""TV"", ""Heating"", ""Kitchen""]",143.0,5,50,2.0,5.0,50.0,50.0,5.0,50.0,True,10,40,66,341,2025-09-15,53,4,1,84,0,40,5720.0,2010-11-01,2025-09-11,4.68,4.71,4.90,4.87,4.81,4.88,4.59,Unknown,True,17,17,0,0,0.29,1.0,bath,15.312799,5432.0,4.0,14.861054,Entire place,high,high,high,0
2,62423,20250914152907,2025-09-15,city scrape,MAGIC ARTISTIC HOUSE IN THE CENTER OF MADRID,"INCREDIBLE HOME OF AN ARTIST SURROUNDED BY PAINTINGS AND ARTWORK. AMPLE AND BRIGHT APARTMENT OF 120 M2 WITH A SPACIOUS, OPEN KITCHEN. PERFECT TO COOK AT HOME AND ENJOY DINNER. FEEL HOME AT THIS WE...","DISTRICT WITH VERY GOOD VIBES IN THE MIDDLE OF MADRID, BESIDE GRAN VÍA AND ALCALÁ (METRO BANCO DE ESPAÑA (METRO LINE 2)<br />VERY GOOD LOCATION AT CHUECA DISTRICT FULL OF RESTAURANTS, MUSEUMS, CIN...",https://a0.muscache.com/pictures/miso/Hosting-62423/original/e6af64ea-0329-408b-b572-e1b3e5c15b5f.jpeg,303845,Arturo,2010-11-29,"Barcelona, Spain","I am an artist... and I will be glad to have you at my Home Art Museum... in a sun

## 2 · Univariate analysis

### 2.1 Numerical distributions

In [3]:
num_vars = [
    "price", "number_of_reviews", "review_scores_rating",
    "accommodates", "calculated_host_listings_count",
]
df_num = l_df[num_vars].dropna()

stats = df_num.describe().T
stats["skewness"] = df_num.skew()
display(stats)

for col in num_vars:
    fig = make_subplots(
        rows=1, cols=2, column_widths=[0.6, 0.4],
        subplot_titles=(f"Histogram — {col}", f"Boxplot — {col}"),
    )
    fig.add_trace(
        go.Histogram(x=df_num[col], nbinsx=40, histnorm="percent", name="Hist"),
        row=1, col=1,
    )
    fig.add_trace(
        go.Box(x=df_num[col], boxmean=True, name="Box"),
        row=1, col=2,
    )
    fig.update_layout(title=f"Univariate: {col}", height=380, showlegend=False)
    fig.show()

,count,mean,std,min,25%,50%,75%,max,skewness
price,15772.0,134.088638,102.629011,8.0,73.00,111.00,162.0,1000.0,2.987664
number_of_reviews,15772.0,71.661869,109.566333,1.0,7.00,29.00,88.0,1184.0,3.031647
review_scores_rating,15772.0,4.631128,0.482064,1.0,4.54,4.75,4.9,5.0,-3.886322
accommodates,15772.0,3.373320,1.947994,1.0,2.00,3.00,4.0,16.0,1.716631
calculated_host_listings_count,15772.0,37.995879,80.674525,1.0,2.00,5.00,25.0,407.0,2.944645


### 2.2 Categorical distributions

In [4]:
cat_vars = [
    "room_type", "property_type_std", "neighbourhood_group_cleansed",
    "host_is_superhost", "instant_bookable", "host_identity_verified",
]
for col in cat_vars:
    counts = l_df[col].value_counts(dropna=False)
    pcts   = counts / counts.sum() * 100

    fig_bar = px.bar(
        x=counts.index.astype(str), y=counts.values,
        labels={"x": col, "y": "Count"}, title=f"Count — {col}",
        text=counts.values,
    )
    fig_bar.update_layout(xaxis_tickangle=-40)
    fig_bar.show()

    fig_pie = px.pie(
        names=pcts.index.astype(str), values=pcts.values,
        title=f"Distribution — {col}", hole=0.0,
    )
    fig_pie.update_traces(textposition="inside", textinfo="percent+label")
    fig_pie.show()

## 3 · Bivariate analysis

### 3.1 Correlation matrix

In [5]:
# Filter out identifiers and metadata column names before correlation analysis
exclude_cols = ['id', 'scrape_id', 'host_id', 'latitude', 'longitude']
cols_to_corr = [c for c in l_df.select_dtypes(include=['int64', 'float64']).columns if c not in exclude_cols]
num_df = l_df[cols_to_corr]
corr   = num_df.corr()

fig = px.imshow(
    corr, text_auto=False, color_continuous_scale='RdBu_r',
    aspect='auto', title='Correlation matrix — numerical features (identifiers excluded)',
)
fig.update_layout(height=700)
fig.show()


### 3.2 Price vs key numerical variables

In [6]:
price_cap = l_df["price"].quantile(0.99)
df_plot   = l_df[l_df["price"] <= price_cap].copy()

for x_col in ["review_scores_rating", "accommodates", "number_of_reviews"]:
    r = l_df["price"].corr(l_df[x_col])
    fig = px.scatter(
        df_plot, x=x_col, y="price", trendline="ols", opacity=0.4,
        title=f"Price vs {x_col}   (Pearson r = {r:.3f})",
    )
    fig.show()

### 3.2.1 Log-Price distribution & normality analysis


In [7]:
# Plot log-price distribution to check skewness vs raw price
stats_price = pd.DataFrame({
    "Raw Price skewness": [l_df["price"].skew()],
    "Log Price (np.log1p) skewness": [np.log1p(l_df["price"]).skew()]
})
display(stats_price)

fig_log = px.histogram(
    l_df, x=np.log1p(l_df["price"]), nbins=40, histnorm="percent",
    title="Log-transformed Price Distribution (np.log1p)",
    labels={"x": "log(Price + 1)"}
)
fig_log.show()


,Raw Price skewness,Log Price (np.log1p) skewness
0,3.187174,0.010052


### 3.3 Price by room type

In [8]:
display(l_df.groupby("room_type")["price"]
        .agg(["mean", "median", "std", "count"]))

fig = px.box(
    l_df, x="room_type", y="price",
    title="Price distribution by room type", points=False,
)
fig.show()

median_prices = l_df.groupby("room_type")["price"].median().reset_index()
fig2 = px.bar(median_prices, x="room_type", y="price",
              title="Median price by room type",
              labels={"price": "Median price (€)"})
fig2.show()

groups = [g["price"].dropna() for _, g in l_df.groupby("room_type")]
f, p = f_oneway(*groups)
print(f"ANOVA: F = {f:.4f},  p = {p:.2e}")

,mean,median,std,count
room_type,,,,
Entire home/apt,158.743427,129.0,109.854861,13579
Hotel room,151.097561,152.0,63.683909,41
Private room,76.259027,50.0,94.284536,5096
Shared room,53.198630,30.0,117.505135,146


ANOVA: F = 779.9341,  p = 0.00e+00


### 3.4 Reviews by neighbourhood (top 15)

In [9]:
top_neigh = l_df["neighbourhood_cleansed"].value_counts().head(15).index
l_top     = l_df[l_df["neighbourhood_cleansed"].isin(top_neigh)]

display(l_top.groupby("neighbourhood_cleansed")["number_of_reviews"]
        .agg(["count", "mean", "median", "std"])
        .sort_values("mean", ascending=False))

fig = px.box(
    l_top, x="neighbourhood_cleansed", y="number_of_reviews",
    title="Number of reviews by neighbourhood (top 15)", points=False,
)
fig.update_layout(width=1100, height=500, xaxis_tickangle=45)
fig.show()

groups = [g["number_of_reviews"].dropna() for _, g in l_top.groupby("neighbourhood_cleansed")]
f, p = f_oneway(*groups)
print(f"ANOVA: F = {f:.4f},  p = {p:.2e}")

,count,mean,median,std
neighbourhood_cleansed,,,,
Palos de Moguer,300,113.596667,47.5,176.460217
Sol,1096,103.350365,40.0,151.342798
Cortes,859,88.024447,35.0,123.392567
Embajadores,1977,83.638847,28.0,127.788738
Palacio,1474,80.905020,31.0,116.722772
Universidad,1702,79.302585,28.0,122.690481
Justicia,940,64.982979,24.5,96.651861
Ibiza,223,51.838565,8.0,93.577597
Guindalera,302,49.360927,10.0,86.441954


ANOVA: F = 24.2825,  p = 6.30e-63


### 3.5 Superhost vs non-superhost

In [10]:
metrics = ["price", "number_of_reviews", "review_scores_rating"]
display(l_df.groupby("host_is_superhost")[metrics]
        .agg(["mean", "median", "std"]))

for m in metrics:
    fig = px.box(l_df, x="host_is_superhost", y=m,
                 title=f"{m} — Superhost vs Non-Superhost", points=False)
    fig.show()

fig_resp = px.histogram(
    l_df, x="host_response_time", color="host_is_superhost",
    barmode="group", title="Response time by superhost status",
    labels={"host_is_superhost": "Superhost"},
)
fig_resp.show()

sh  = l_df[l_df["host_is_superhost"]]["price"].dropna()
nsh = l_df[~l_df["host_is_superhost"]]["price"].dropna()
t, p = ttest_ind(sh, nsh, equal_var=False)
print(f"Welch t-test on price: t = {t:.4f},  p = {p:.4f}")

price                    number_of_reviews         \
                         mean median         std              mean median   
host_is_superhost                                                           
False              135.368112  109.0  113.994985         44.805851   10.0   
True               136.388702  113.0  106.784673        104.907673   57.0   

                              review_scores_rating                   
                          std                 mean median       std  
host_is_superhost                                                    
False               87.122002             4.544281   4.67  0.538971  
True               131.981119             4.841406   4.87  0.170996

Welch t-test on price: t = 0.5598,  p = 0.5756


## 4 · Multivariate analysis

### 4.1 Price category × occupancy & revenue

In [11]:
cat_order = ["low", "medium", "high", "very_high"]

if "estimated_occupancy_l365d" in l_df.columns:
    display(l_df.groupby("price_cat", observed=True)["estimated_occupancy_l365d"]
            .median().reindex(cat_order))
    fig = px.box(l_df, x="price_cat", y="estimated_occupancy_l365d",
                 category_orders={"price_cat": cat_order},
                 title="Estimated occupancy (365 d) by price category",
                 points=False)
    fig.show()

if "estimated_revenue_l365d" in l_df.columns:
    rev_df = l_df.dropna(subset=["estimated_revenue_l365d"])
    display(rev_df.groupby("price_cat", observed=True)["estimated_revenue_l365d"]
            .median().reindex(cat_order))
    fig = px.box(rev_df, x="price_cat", y="estimated_revenue_l365d",
                 category_orders={"price_cat": cat_order},
                 title="Estimated revenue (365 d) by price category",
                 points=False)
    fig.show()

price_cat
low          54.0
medium       90.0
high         78.0
very_high    54.0
Name: estimated_occupancy_l365d, dtype: float64

price_cat
low           2310.0
medium        7884.0
high         10062.0
very_high    12000.0
Name: estimated_revenue_l365d, dtype: float64

### 4.2 Property type × neighbourhood

In [12]:
top_neigh = l_df["neighbourhood_cleansed"].value_counts().head(15).index
l_sub = l_df[l_df["neighbourhood_cleansed"].isin(top_neigh)]

# Stacked proportional bar
ct = pd.crosstab(l_sub["neighbourhood_cleansed"], l_sub["property_type_std"])
ct_prop = ct.div(ct.sum(axis=1), axis=0)

fig = px.bar(
    ct_prop.reset_index().melt(
        id_vars="neighbourhood_cleansed",
        var_name="property_type_std", value_name="proportion",
    ),
    x="neighbourhood_cleansed", y="proportion",
    color="property_type_std",
    title="Property type mix by neighbourhood",
    barmode="stack",
)
fig.update_layout(height=600, width=1100, xaxis_tickangle=45)
fig.show()

# Room type heatmap across district groups
ct2 = pd.crosstab(l_df["neighbourhood_group_cleansed"], l_df["room_type"])
fig2 = px.imshow(
    ct2[["Entire home/apt", "Private room"]],
    title="Entire home vs private room by district",
    text_auto=True, aspect="auto",
)
fig2.update_layout(height=500, width=700)
fig2.show()

### 4.3 3-D scatter: review score × price × review volume

In [13]:
price_cap = l_df["price"].quantile(0.99)
df_no_out = l_df[l_df["price"] <= price_cap].copy()

fig = px.scatter_3d(
    df_no_out,
    x="review_scores_rating", y="price", z="number_of_reviews",
    color="property_type_std", size="number_of_reviews", opacity=0.6,
    title="Review score × Price × Review volume (outliers removed)",
    labels={"review_scores_rating": "Review Score",
            "price": "Price (€)", "number_of_reviews": "# Reviews"},
)
fig.update_layout(height=750, width=1000)
fig.show()

### 4.1 Barcelona District Performance Quadrants (Price vs. Occupancy)


In [14]:
# Group by district and calculate median price and occupancy
dist_stats = l_df.groupby("neighbourhood_group_cleansed").agg(
    median_price = ("price", "median"),
    median_occupancy = ("estimated_occupancy_l365d", "median")
).reset_index()

# Thresholds based on medians of district-level medians
price_threshold = dist_stats["median_price"].median()
occupancy_threshold = dist_stats["median_occupancy"].median()

fig_quad = px.scatter(
    dist_stats, x="median_price", y="median_occupancy",
    text="neighbourhood_group_cleansed", size="median_price",
    title="Barcelona District Performance Quadrants (Price vs. Occupancy)",
    labels={"median_price": "Median Price (€)", "median_occupancy": "Median Occupancy (Days/Year)"}
)
fig_quad.add_hline(y=occupancy_threshold, line_dash="dash", annotation_text="Occupancy Median")
fig_quad.add_vline(x=price_threshold, line_dash="dash", annotation_text="Price Median")
fig_quad.update_traces(textposition="top center")
fig_quad.update_layout(height=600, width=1000)
fig_quad.show()


### 4.2 Host Portfolio Scale Analysis


In [15]:
# Segment hosts by their portfolio scale
def host_scale(x):
    if x == 1:
        return "1 (Peer-to-Peer)"
    if x <= 5:
        return "2-5 (Small Portfolio)"
    return "6+ (Commercial Operator)"

l_df["host_portfolio_scale"] = l_df["calculated_host_listings_count"].apply(host_scale)
scale_summary = l_df.groupby("host_portfolio_scale").agg(
    count = ("price", "count"),
    median_price = ("price", "median"),
    median_occupancy = ("estimated_occupancy_l365d", "median")
).reset_index()
scale_summary["percentage"] = scale_summary["count"] / scale_summary["count"].sum() * 100
display(scale_summary)

fig_scale = px.bar(
    scale_summary, x="host_portfolio_scale", y="percentage",
    text=scale_summary["percentage"].apply(lambda x: f"{x:.1f}%"),
    title="Barcelona Host Portfolio Scale Distribution"
)
fig_scale.show()


,host_portfolio_scale,count,median_price,median_occupancy,percentage
0,1 (Peer-to-Peer),4435,95.0,96.0,23.512883
1,2-5 (Small Portfolio),4620,98.0,96.0,24.493691
2,6+ (Commercial Operator),9807,121.0,48.0,51.993426


### 4.3 Amenity Association Analysis (Apriori rules)


In [16]:
# Pure Python implementation of Apriori association rule mining for top 20 amenities
import json
from collections import Counter

def parse_amenities(x):
    if not isinstance(x, str):
        return []
    try:
        return json.loads(x)
    except:
        return re.findall(r'"([^"]*)"', x)

amenity_lists = l_df["amenities"].apply(parse_amenities)
all_amenities = [a for ams in amenity_lists for a in ams]
top_amenities = [item for item, count in Counter(all_amenities).most_common(20)]

# Construct binary matrix
binary_matrix = pd.DataFrame()
for amen in top_amenities:
    binary_matrix[amen] = amenity_lists.apply(lambda x: amen in x)

# Compute rules
rules = []
for i, amen_a in enumerate(top_amenities):
    support_a = binary_matrix[amen_a].mean()
    for j, amen_b in enumerate(top_amenities):
        if i == j:
            continue
        support_b = binary_matrix[amen_b].mean()
        support_ab = (binary_matrix[amen_a] & binary_matrix[amen_b]).mean()
        
        if support_a > 0 and support_b > 0:
            confidence = support_ab / support_a
            lift = support_ab / (support_a * support_b)
            if lift > 1.05 and confidence > 0.4:
                rules.append({
                    "Rule": f"{amen_a} -> {amen_b}",
                    "Support A": support_a,
                    "Support B": support_b,
                    "Support A&B": support_ab,
                    "Confidence": confidence,
                    "Lift": lift
                })

rules_df = pd.DataFrame(rules).sort_values("Lift", ascending=False).head(15)
print("Top 15 Amenity Association Rules (Apriori):")
display(rules_df)


Top 15 Amenity Association Rules (Apriori):


,Rule,Support A,Support B,Support A&B,Confidence,Lift
178,Heating -> Coffee maker,0.584349,0.451384,0.361839,0.619216,1.371818
232,Coffee maker -> Heating,0.451384,0.584349,0.361839,0.801621,1.371818
210,Freezer -> Microwave,0.491517,0.688209,0.455201,0.926114,1.345686
105,Microwave -> Freezer,0.688209,0.491517,0.455201,0.661428,1.345686
121,Refrigerator -> Coffee maker,0.675485,0.451384,0.407062,0.602621,1.335054
226,Coffee maker -> Refrigerator,0.451384,0.675485,0.407062,0.901809,1.335054
211,Freezer -> Refrigerator,0.491517,0.675485,0.437016,0.889117,1.316264
120,Refrigerator -> Freezer,0.675485,0.491517,0.437016,0.646966,1.316264
62,Dishes and silverware -> Freezer,0.718694,0.491517,0.464055,0.645692,1.313671
207,Freezer -> Dishes and silverware,0.491517,0.718694,0.464055,0.944127,1.313671


### 4.4 K-means market segmentation


In [17]:
feat_cols = ["price", "availability_365", "review_scores_rating", "number_of_reviews"]
features  = l_df[feat_cols].dropna().copy()
scaled    = StandardScaler().fit_transform(features)

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
features["cluster"] = kmeans.fit_predict(scaled)

fig = px.scatter(
    features, x="price", y="availability_365", color="cluster",
    title="Market segments — Price × Availability (k = 4)",
)
fig.show()

l_df = l_df.merge(features[["cluster"]], left_index=True, right_index=True, how="left")

segment_summary = l_df.groupby("cluster").agg(
    count          = ("price", "count"),
    median_price   = ("price", "median"),
    median_avail   = ("availability_365", "median"),
    median_reviews = ("number_of_reviews", "median"),
    median_rating  = ("review_scores_rating", "median"),
)
display(segment_summary)

/Users/nuria/Documents/IE/CAPSTONE/airbnb-investment-intelligence/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/nuria/Documents/IE/CAPSTONE/airbnb-investment-intelligence/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/nuria/Documents/IE/CAPSTONE/airbnb-investment-intelligence/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/nuria/Documents/IE/CAPSTONE/airbnb-investment-intelligence/.venv/lib/python3.9/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning: divide by zero encountered in matmul
  current_pot = closest_dist_sq @ sample_weight
/Users/nuria/Documents/IE/CAPSTONE/airbnb-investment-intelligence/.venv/lib/python3.9/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning: overflow encountered in matmul
  cu

,count,median_price,median_avail,median_reviews,median_rating
cluster,,,,,
0.0,8153,115.0,304.0,21.0,4.75
1.0,5668,107.0,75.0,32.0,4.78
2.0,1379,115.0,161.0,318.0,4.78
3.0,572,95.0,289.0,2.0,3.00


## 5 · Key findings

_Fill in the most important insights discovered in this notebook._